In [1]:
import torch
from datasets import Dataset
from modelscope import snapshot_download, AutoTokenizer
from swanlab.integration.transformers import SwanLabCallback
from qwen_vl_utils import process_vision_info
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
)
import swanlab
import json

D:\toolkit\anaconda\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# # 在modelscope上下载Qwen2.5-VL模型到本地目录下
# model_dir = snapshot_download("Qwen/Qwen2.5-VL-7B-Instruct", cache_dir="./", revision="master")
cache_dir = "D:/cache/huggingface"
model_path = "Qwen/Qwen2.5-VL-7B-Instruct"
# 使用Transformers加载模型权重
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False, trust_remote_code=True, cache_dir=cache_dir+"/models")
processor = AutoProcessor.from_pretrained(model_path, cache_dir=cache_dir+"/models")
# 加载 Qwen2.5-VL-7B-Instruct
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto",
    cache_dir=cache_dir + "/models"
)


2025-12-06 21:42:33,151 - modelscope - INFO - Creating symbolic link [C:\Users\yitin\.cache\modelscope\hub\models\Qwen\Qwen2.5-VL-7B-Instruct].
2025-12-06 21:42:33,151 - modelscope - WARNING - Failed to create symbolic link C:\Users\yitin\.cache\modelscope\hub\models\Qwen\Qwen2.5-VL-7B-Instruct for C:\Users\yitin\.cache\modelscope\hub\models\Qwen\Qwen2___5-VL-7B-Instruct.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:07<00:00,  1.51s/it]


In [3]:
# ====================测试模式===================
# 配置测试参数
val_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    inference_mode=True,  # 测试模式
    r=16,  # Lora 秩
    lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1,  # Dropout 比例
    bias="none",
)
# 获取测试模型
val_peft_model = PeftModel.from_pretrained(model, model_id="./output/Qwen2.5-VL-7B-nutrition/checkpoint-215", config=val_config)
val_peft_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2_5_VLForConditionalGeneration(
      (model): Qwen2_5_VLModel(
        (visual): Qwen2_5_VisionTransformerPretrainedModel(
          (patch_embed): Qwen2_5_VisionPatchEmbed(
            (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
          )
          (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-31): 32 x Qwen2_5_VLVisionBlock(
              (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
              (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
              (attn): Qwen2_5_VLVisionAttention(
                (qkv): Linear(in_features=1280, out_features=3840, bias=True)
                (proj): Linear(in_features=1280, out_features=1280, bias=True)
              )
              (mlp): Qwen2_5_VLMLP(
                (gate_proj): lora.Linear(
                  (base_layer): Linear(in_features=1280, out_features=3420, bias=True)
               

In [4]:
def predict(messages, model):
    # 加上這個，告訴 PyTorch 不需要計算梯度
    with torch.no_grad():
        # 准备推理
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")

        # 生成输出
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )

        # 釋放顯存 (可選)
        del inputs
        del generated_ids
        torch.cuda.empty_cache()

        return output_text[0]

In [20]:
image_path = "./data/img/burger.png"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1])

{'role': 'assistant', 'content': 'Name: Cheeseburger with Bacon and Pickles\n\nMain Ingredients: cheeseburgers (201.0g), pickles (35.0g), onions (28.0g)\n\nNutritional Information (per serving):\n- Calories: 649.7 kcal\n- Protein: 31.0 g\n- Fat: 36.6 g\n- Carbohydrates: 30.0 g\n- Total Mass: 264.0 g\n\nNutritional Analysis: This dish is relatively high in calories and fat, primarily from the cheeseburger patty and bacon. The protein content is moderate. Consider the sodium content due to the pickles and bacon, especially for individuals monitoring their sodium intake.\n'}


In [7]:
image_path = "./data/img/fuqifeipian.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1])

{'role': 'assistant', 'content': 'Name: Beef and Pork Noodle Bowl with Vegetables\n\nMain Ingredients: beef (148.0g), pork (97.0g), noodles (53.0g), cabbage (26.5g), bok choy (26.5g)\n\nNutritional Information (per serving):\n- Calories: 810.1 kcal\n- Protein: 68.4 g\n- Fat: 40.6 g\n- Carbohydrates: 23.8 g\n- Total Mass: 382.0 g\n\nNutritional Analysis: This dish is high in protein, primarily from beef and pork, contributing significantly to its caloric content. The fat content is also relatively high. Fiber is present from vegetables like cabbage and bok choy, but the carbohydrate content is moderate.\n'}


In [8]:
image_path = "./data/img/laziji.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1])

{'role': 'assistant', 'content': 'Name: Chicken and Chayote Stir-fry with Sesame Seeds\n\nMain Ingredients: chicken (128.0g), chayote squash (96.0g), chili peppers (74.0g), white rice (35.0g)\n\nNutritional Information (per serving):\n- Calories: 349.2 kcal\n- Protein: 35.8 g\n- Fat: 13.8 g\n- Carbohydrates: 19.9 g\n- Total Mass: 359.0 g\n\nNutritional Analysis: This dish provides a significant amount of protein relative to its caloric content, primarily from the chicken. The fat content is moderate. Fiber is contributed by the vegetables and chayote squash.\n'}


In [9]:
image_path = "./data/img/macarmoons.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1])

{'role': 'assistant', 'content': 'Name: Macaron (Single Serving)\n\nMain Ingredients: macarons (58.0g)\n\nNutritional Information (per serving):\n- Calories: 241.6 kcal\n- Protein: 2.7 g\n- Fat: 13.9 g\n- Carbohydrates: 26.9 g\n- Total Mass: 58.0 g\n\nNutritional Analysis: This macaron serving is relatively high in fat and carbohydrates, contributing significantly to its caloric content. The protein-to-calorie ratio is low. Consider portion control due to the high calorie density.\n'}


In [10]:
image_path = "./data/img/cheesecake.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1])

{'role': 'assistant', 'content': 'Name: Strawberry Cheesecake with Berry Sauce\n\nMain Ingredients: strawberries (147.0g), cream cheese (135.0g), graham crackers (126.0g), sugar (89.0g)\n\nNutritional Information (per serving):\n- Calories: 809.0 kcal\n- Protein: 11.7 g\n- Fat: 42.6 g\n- Carbohydrates: 96.8 g\n- Total Mass: 522.0 g\n\nNutritional Analysis: This dessert is high in calories and carbohydrates, primarily from sugars and refined grains. The protein content is relatively low compared to the caloric value. Consider portion control due to the high carbohydrate and fat content.\n'}


In [11]:
image_path = "./data/img/pasta.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1])

{'role': 'assistant', 'content': 'Name: Penne with Tomato Basil Sauce\n\nMain Ingredients: penne pasta (305.0g), tomato sauce (126.0g), parmesan cheese (49.0g)\n\nNutritional Information (per serving):\n- Calories: 781.3 kcal\n- Protein: 27.6 g\n- Fat: 26.8 g\n- Carbohydrates: 109.7 g\n- Total Mass: 480.0 g\n\nNutritional Analysis: This dish provides a moderate amount of protein relative to its caloric content. The fat content is notable, and the carbohydrate source is primarily from pasta. Consider adding more fiber-rich vegetables to enhance the nutritional value.\n'}


In [25]:
image_path = "./data/img/milkshake.jpg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1])

{'role': 'assistant', 'content': 'Name: Oreo Milkshake\n\nMain Ingredients: milkshakes (530.0g), oreos (128.0g)\n\nNutritional Information (per serving):\n- Calories: 946.7 kcal\n- Protein: 24.3 g\n- Fat: 46.0 g\n- Carbohydrates: 110.2 g\n- Total Mass: 658.0 g\n\nNutritional Analysis: This milkshake is high in calories and carbohydrates, with a moderate amount of fat. The protein content is relatively low compared to the caloric value. Consider portion control due to the high sugar content derived from the oreos.\n'}


In [12]:
image_path = "./data/img/cats.jpeg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Generate a nutrition report for the given food item:",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1])

{'role': 'assistant', 'content': 'Name: Chicken and Rice Bowl with Vegetables\n\nMain Ingredients: chicken (102.0g), white rice (54.0g), broccoli (36.9g), carrot (36.9g)\n\nNutritional Information (per serving):\n- Calories: 278.1 kcal\n- Protein: 25.6 g\n- Fat: 10.6 g\n- Carbohydrates: 24.2 g\n- Total Mass: 268.0 g\n\nNutritional Analysis: This dish provides a good source of protein relative to its caloric content. The fat content is moderate, while carbohydrates are derived from rice and vegetables. Consider increasing fiber intake by adding more non-starchy vegetables.\n'}


In [19]:
image_path = "./data/img/cats.jpeg"
messages = [{
    "role": "user",
    "content": [
        {
        "type": "image",
        "image": image_path
        },
        {
        "type": "text",
        "text": "Describe the image",
        }
    ]}]

response = predict(messages, val_peft_model)
messages.append({"role": "assistant", "content": f"{response}"})
print(messages[-1])

{'role': 'assistant', 'content': "The image shows six kittens with a light brown and white coat, sitting closely together on a soft surface. They all have large, expressive eyes that appear to be looking directly at the camera. The kittens' fur is fluffy, and their ears are perked up, giving them an alert appearance. The background is simple and does not distract from the kittens. The overall mood of the image is warm and endearing, highlighting the cuteness of the kittens."}
